In [12]:
import pandas as pd
import numpy as np
import shap
import joblib
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

ind_hosp = pd.read_parquet('ind_hosp.parquet')
calibrated = joblib.load('lightgbm.pkl')
base_model = calibrated.calibrated_classifiers_[0].estimator

cols_to_drop = ['subject_id', 'hadm_id', 'dischtime', 'current_date', 
                'target_readmission_30d', 'los']
cols_to_drop = [c for c in cols_to_drop if c in ind_hosp.columns]
X = ind_hosp.drop(columns=cols_to_drop)

bool_cols = X.select_dtypes(include=['bool']).columns
if len(bool_cols) > 0:
    X[bool_cols] = X[bool_cols].astype(int)

patient_target = ind_hosp.groupby('subject_id')['target_readmission_30d'].max().reset_index()
patient_target.columns = ['subject_id', 'has_readmission']

train_val_ids, test_ids = train_test_split(
    patient_target['subject_id'],
    test_size=0.2,
    random_state=42,
    stratify=patient_target['has_readmission']
)
test_patients = test_ids.tolist() 
# SAMPLE_SIZE = 1000

# test_patients_sample, _ = train_test_split(
#     test_ids,
#     train_size=SAMPLE_SIZE,
#     random_state=42,
#     stratify=patient_target[patient_target['subject_id'].isin(test_ids)]['has_readmission']
# )
# test_patients = test_patients_sample.tolist()

print(f"Test patients (SHAP): {len(test_patients)}")

test_mask = ind_hosp['subject_id'].isin(test_patients)
X_sample = X[test_mask].copy()
y_sample = ind_hosp.loc[test_mask, 'target_readmission_30d'].copy()

/Users/arinapetuhova/.pyenv/versions/3.13.7/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Test patients (SHAP): 35915


In [13]:
from tqdm import tqdm
print('Calculating SHAP...')
explainer = shap.TreeExplainer(base_model)
batch_size = 1000
n_batches = int(np.ceil(len(X_sample) / batch_size))

shap_values_list = []

for i in tqdm(range(n_batches), desc="Computing SHAP"):
    start_idx = i * batch_size
    end_idx = min((i + 1) * batch_size, len(X_sample))
    X_batch = X_sample.iloc[start_idx:end_idx]
    
    shap_batch = explainer.shap_values(X_batch)
    shap_values_list.append(shap_batch)

shap_values = np.vstack(shap_values_list)
print(f"SHAP values shape: {shap_values.shape}")
print('Done')

Calculating SHAP...


Computing SHAP: 100%|██████████| 366/366 [07:56<00:00,  1.30s/it]


SHAP values shape: (365442, 132)
Done


In [14]:
explanation = shap.Explanation(
    values=shap_values,
    base_values=explainer.expected_value,
    data=X_sample.values,
    feature_names=X_sample.columns.tolist()
)
joblib.dump(explanation, 'shap_explanation.pkl')

['shap_explanation.pkl']

In [15]:
hadm_ids = ind_hosp.loc[X_sample.index, 'hadm_id'].values
shap_df_full = pd.DataFrame(
    shap_values,
    columns=X_sample.columns,
    index=X_sample.index
)
shap_df_full['hadm_id'] = hadm_ids

shap_by_hadm = shap_df_full.groupby('hadm_id').mean()
shap_abs_by_day = np.abs(shap_values)
shap_abs_df = pd.DataFrame(
    shap_abs_by_day,
    columns=X_sample.columns,
    index=X_sample.index
)
shap_abs_df['hadm_id'] = hadm_ids
shap_abs_by_hadm = shap_abs_df.groupby('hadm_id').mean()

shap_mean_original = shap_by_hadm.mean(axis=0)
shap_mean_abs = shap_abs_by_hadm.mean(axis=0)

icd_cols = [col for col in X.columns if col.startswith('icd_')]
ccsr_cols = [col for col in X.columns if col.startswith('ccsr_')]
lab_cols = [col for col in X.columns if col.startswith('lab_') and col.endswith('_daily')]

features_to_analyze = (
    icd_cols +
    ccsr_cols +
    lab_cols +
    [
        'num_diagnoses',
        'num_chronic',
        'comorbidity_score',
        'num_medications_daily',
        'prior_admissions_12mo',
        'cumulative_procedures',
        'cumulative_medications',
        'num_procedures_daily',
        'gender_male',
        'age'
    ]
)

shap_by_hadm_reset = shap_by_hadm.reset_index()
shap_by_hadm_reset.to_csv('shap_matrix.csv', index=False)

In [16]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

delta_summary = pd.read_csv('delta_summary.csv')
shap_matrix = pd.read_csv('shap_matrix.csv')
if 'hadm_id' in shap_matrix.columns:
    shap_matrix = shap_matrix.drop('hadm_id', axis=1)

shap_means = shap_matrix.mean(axis=0)
shap_abs_means = np.abs(shap_matrix).mean(axis=0)
shap_std = shap_matrix.std(axis=0)

shap_df = pd.DataFrame({
    'feature': shap_matrix.columns,
    'shap_mean': shap_means.values,
    'shap_abs_mean': shap_abs_means.values,
    'shap_std': shap_std.values
})

delta_cols = ['feature', 'delta_mean', 'delta_abs_mean', 'delta_std', 'most_common_direction']
comparison = shap_df.merge(
    delta_summary[delta_cols],
    on='feature',
    how='inner'
)

In [17]:
spearman_mean = spearmanr(comparison['delta_mean'], comparison['shap_mean'])[0]
spearman_abs = spearmanr(comparison['delta_abs_mean'], comparison['shap_abs_mean'])[0]

print(f"Spearman (Δ vs SHAP_mean):    {spearman_mean:.4f}")
print(f"Spearman (|Δ| vs SHAP_abs):   {spearman_abs:.4f}")

comparison['delta_dir'] = np.sign(comparison['delta_mean'])
comparison['shap_dir'] = np.sign(comparison['shap_mean'])
comparison['direction_agreement'] = comparison['delta_dir'] == comparison['shap_dir']
agreement_rate = comparison['direction_agreement'].mean()
print(f"Sign direction agreement rate: {agreement_rate:.2%}")

Spearman (Δ vs SHAP_mean):    0.0695
Spearman (|Δ| vs SHAP_abs):   0.8501
Sign direction agreement rate: 50.00%


In [18]:
lab_names = {
    '50983': 'Sodium',
    '50971': 'Potassium',
    '50902': 'Chloride',
    '50882': 'Bicarbonate',
    '50912': 'Creatinine',
    '51006': 'BUN',
    '50931': 'Glucose',
    '50893': 'Calcium',
    '50868': 'Anion Gap',
    '51222': 'Hemoglobin',
    '51301': 'WBC',
    '51265': 'Platelet Count',
    '51221': 'Hematocrit',
    '51250': 'MCV',
    '51277': 'RDW',
    '50960': 'Magnesium',
    '50970': 'Phosphate',
    '51248': 'MCH',
    '51249': 'MCHC',
    '51279': 'RBC'
}

def map_lab_name(feature_name):
    if feature_name.startswith('lab_') and feature_name.endswith('_daily'):
        code = feature_name.replace('lab_', '').replace('_daily', '')
        if code in lab_names:
            return f"{lab_names[code]} ({code})"
    return feature_name

comparison['feature'] = comparison['feature'].apply(map_lab_name)

In [19]:
icd_names = {
    'I10': 'Essential (primary) hypertension',
    'E785': 'Hyperlipidemia, unspecified',
    'K219': 'Gastroesophageal reflux disease without esophagitis',
    'Z87891': 'Personal history of nicotine dependence',
    'I2510': 'Atherosclerotic heart disease of native coronary artery without angina pectoris',
    'N179': 'Acute kidney failure, unspecified',
    'F329': 'Major depressive disorder, single episode, unspecified',
    'I4891': 'Unspecified atrial fibrillation',
    'Z7901': 'Long term (current) use of anticoagulants',
    'F419': 'Anxiety disorder, unspecified',
    'E119': 'Type 2 diabetes mellitus without complications',
    'E039': 'Hypothyroidism, unspecified',
    'Z794': 'Long term (current) use of insulin',
    'D649': 'Anemia, unspecified',
    'N390': 'Urinary tract infection, site not specified'
}

ccsr_names = {
    'FAC021': 'Personal/family history of disease',
    'FAC025': 'Other specified status',
    'END011': 'Fluid and electrolyte disorders',
    'CIR011': 'Coronary atherosclerosis and other heart disease',
    'END010': 'Disorders of lipid metabolism',
    'CIR007': 'Essential hypertension',
    'END003': 'Diabetes mellitus with complication',
    'CIR019': 'Heart failure',
    'DIG004': 'Esophageal disorders',
    'CIR017': 'Cardiac dysrhythmias',
    'CIR008': 'Hypertension with complications and secondary hypertension',
    'BLD003': 'Aplastic anemia',
    'EXT027': 'External cause codes: place of occurrence of the external cause',
    'GEN002': 'Acute and unspecified renal failure',
    'END009': 'Obesity'
}

def map_icd_name(feature_name):
    if feature_name.startswith('icd_'):
        code = feature_name.replace('icd_', '')
        if code in icd_names:
            return f"{icd_names[code]} ({code})"
    return feature_name

def map_ccsr_name(feature_name):
    if feature_name.startswith('ccsr_'):
        code = feature_name.replace('ccsr_', '')
        if code in ccsr_names:
            return f"{ccsr_names[code]} ({code})"
    return feature_name

comparison['feature'] = comparison['feature'].apply(map_icd_name)
comparison['feature'] = comparison['feature'].apply(map_ccsr_name)

In [20]:
comparison['delta_rank'] = comparison['delta_abs_mean'].rank(ascending=False)
comparison['shap_rank'] = comparison['shap_abs_mean'].rank(ascending=False)
comparison['rank_diff'] = comparison['delta_rank'] - comparison['shap_rank']

results_df = pd.DataFrame({
    'feature': comparison['feature'],
    'delta': comparison['delta_mean'],
    'shap': comparison['shap_mean'],
    'delta_abs': comparison['delta_abs_mean'],
    'shap_abs': comparison['shap_abs_mean'],
    'delta_rank': comparison['delta_rank'],
    'shap_rank': comparison['shap_rank'],
    'rank_diff': comparison['rank_diff'],
    'most_common_direction': comparison.get('most_common_direction', 'unknown')
})

results_df['delta_dir'] = results_df['delta'].apply(lambda x: '↑' if x > 0 else '↓')
results_df['shap_dir'] = results_df['shap'].apply(lambda x: '↑' if x > 0 else '↓')
results_df['dir_match'] = results_df['delta_dir'] == results_df['shap_dir']

results_df = results_df.sort_values('rank_diff', ascending=False)

In [21]:
top_delta_positive = results_df.nlargest(10, 'delta')
top_delta_negative = results_df.nsmallest(10, 'delta')

print(top_delta_positive[['feature', 'delta', 'shap', 'delta_rank', 'shap_rank', 'most_common_direction']].to_string(index=False))
print()
print(top_delta_negative[['feature', 'delta', 'shap', 'delta_rank', 'shap_rank', 'most_common_direction']].to_string(index=False))

                                                                 feature  delta      shap  delta_rank  shap_rank most_common_direction
                                                   prior_admissions_12mo 0.0292 -0.035897         1.0        1.0                   add
                                                       comorbidity_score 0.0235 -0.041818         2.0        2.0                remove
                                                             RDW (51277) 0.0149 -0.013754         3.0        4.0                normal
                                                             RBC (51279) 0.0108 -0.006925         9.0       12.0                   low
                                                  Heart failure (CIR019) 0.0102  0.003840         5.0       13.0                   add
                                                      Hematocrit (51221) 0.0097 -0.007197        11.0       10.0                   low
External cause codes: place of occurrence of the extern

In [25]:
print("|SHAP| top 25:")

top_shap = results_df.nlargest(25, 'shap_abs')
for _, row in top_shap.iterrows():
    print(f"  {row['feature']}: |SHAP| = {row['shap_abs']:.4f} (SHAP = {row['shap']:+.4f}, Δ = {row['delta']:+.4f}) [{row['most_common_direction']}]")

print("|Δ| top 25:")
top_delta = results_df.nlargest(25, 'delta_abs')
for _, row in top_delta.iterrows():
    print(f"  {row['feature']}: |Δ| = {row['delta_abs']:.4f} (Δ = {row['delta']:+.4f}, SHAP = {row['shap']:+.4f}) [{row['most_common_direction']}]")

|SHAP| top 25:
  prior_admissions_12mo: |SHAP| = 0.3091 (SHAP = -0.0359, Δ = +0.0292) [add]
  comorbidity_score: |SHAP| = 0.1546 (SHAP = -0.0418, Δ = +0.0235) [remove]
  num_chronic: |SHAP| = 0.0627 (SHAP = -0.0308, Δ = -0.0119) [increase_to_mean]
  RDW (51277): |SHAP| = 0.0577 (SHAP = -0.0138, Δ = +0.0149) [normal]
  age: |SHAP| = 0.0559 (SHAP = +0.0042, Δ = -0.0035) [older]
  num_diagnoses: |SHAP| = 0.0488 (SHAP = -0.0272, Δ = -0.0132) [increase_to_mean]
  cumulative_procedures: |SHAP| = 0.0292 (SHAP = -0.0109, Δ = -0.0047) [increase_to_mean]
  Sodium (50983): |SHAP| = 0.0260 (SHAP = -0.0091, Δ = +0.0042) [normal]
  num_medications_daily: |SHAP| = 0.0246 (SHAP = -0.0055, Δ = -0.0116) [remove]
  Hematocrit (51221): |SHAP| = 0.0245 (SHAP = -0.0072, Δ = +0.0097) [low]
  Platelet Count (51265): |SHAP| = 0.0232 (SHAP = -0.0054, Δ = -0.0019) [normal]
  RBC (51279): |SHAP| = 0.0230 (SHAP = -0.0069, Δ = +0.0108) [low]
  Heart failure (CIR019): |SHAP| = 0.0177 (SHAP = +0.0038, Δ = +0.0102) [a

In [26]:
top_shap_features = set(results_df.nlargest(25, 'shap_abs')['feature'])
top_delta_features = set(results_df.nlargest(25, 'delta_abs')['feature'])
intersection = top_shap_features & top_delta_features

print(f"Intersection: {len(intersection)} features ({len(intersection)/25*100:.1f}% of top-25)")

Intersection: 22 features (88.0% of top-25)


In [27]:
import pandas as pd
import joblib

clinical_ranges = {
    'lab_50983_daily': {'low': 135, 'high': 147, 'norm': 141},
    'lab_50971_daily': {'low': 3.3, 'high': 5.1, 'norm': 4.2},
    'lab_50902_daily': {'low': 96, 'high': 108, 'norm': 102},
    'lab_50882_daily': {'low': 22, 'high': 32, 'norm': 27},
    'lab_50912_daily': {'low': 0.5, 'high': 1.2, 'norm': 0.85},
    'lab_51006_daily': {'low': 6, 'high': 20, 'norm': 13},
    'lab_50931_daily': {'low': 70, 'high': 100, 'norm': 85},
    'lab_50893_daily': {'low': 8.4, 'high': 10.3, 'norm': 9.35},
    'lab_50868_daily': {'low': 10, 'high': 18, 'norm': 14},
    'lab_51222_daily': {'low': 13.7, 'high': 17.5, 'norm': 15.6},
    'lab_51301_daily': {'low': 4, 'high': 10, 'norm': 7},
    'lab_51265_daily': {'low': 150, 'high': 400, 'norm': 275},
    'lab_51221_daily': {'low': 40, 'high': 51, 'norm': 45.5},
    'lab_51250_daily': {'low': 82, 'high': 98, 'norm': 90},
    'lab_51277_daily': {'low': 10.5, 'high': 15.5, 'norm': 13},
    'lab_50960_daily': {'low': 1.6, 'high': 2.6, 'norm': 2.1},
    'lab_50970_daily': {'low': 2.7, 'high': 4.5, 'norm': 3.6},
    'lab_51248_daily': {'low': 26, 'high': 32, 'norm': 29},
    'lab_51249_daily': {'low': 32, 'high': 37, 'norm': 34.5},
    'lab_51279_daily': {'low': 4.6, 'high': 6.1, 'norm': 5.35}
}

def get_lab_name(feature):
    code = feature.replace('lab_', '').replace('_daily', '')
    return lab_names.get(code, code)

ind_hosp = pd.read_parquet('ind_hosp.parquet')
calibrated = joblib.load('lightgbm.pkl')
base_model = calibrated.calibrated_classifiers_[0].estimator

cols_to_drop = ['subject_id', 'hadm_id', 'dischtime', 'current_date', 
                'target_readmission_30d', 'los']
cols_to_drop = [c for c in cols_to_drop if c in ind_hosp.columns]
X = ind_hosp.drop(columns=cols_to_drop)

bool_cols = X.select_dtypes(include=['bool']).columns
if len(bool_cols) > 0:
    X[bool_cols] = X[bool_cols].astype(int)

data = []
for feature, ranges in clinical_ranges.items():
    pop_mean = X[feature].mean()
    norm = ranges['norm']
    low, high = ranges['low'], ranges['high']
    
    diff = pop_mean - norm
    
    data.append({
        'Feature': get_lab_name(feature.replace('lab_', '').replace('_daily', '')),
        'Norm': norm,
        'Mean': round(pop_mean, 2),
        'Diff': f"{diff:+.1f}",
    })

df = pd.DataFrame(data).sort_values('Mean', ascending=False)
print(df.to_string(index=False))

       Feature   Norm   Mean  Diff
Platelet Count 275.00 238.06 -36.9
        Sodium 141.00 138.30  -2.7
       Glucose  85.00 125.58 +40.6
      Chloride 102.00 102.20  +0.2
           MCV  90.00  90.88  +0.9
          MCHC  34.50  32.61  -1.9
    Hematocrit  45.50  32.12 -13.4
           MCH  29.00  29.63  +0.6
   Bicarbonate  27.00  25.44  -1.6
           BUN  13.00  22.67  +9.7
           RDW  13.00  15.19  +2.2
     Anion Gap  14.00  13.12  -0.9
    Hemoglobin  15.60  10.48  -5.1
           WBC   7.00   9.23  +2.2
       Calcium   9.35   8.68  -0.7
     Potassium   4.20   4.11  -0.1
           RBC   5.35   3.56  -1.8
     Phosphate   3.60   3.48  -0.1
     Magnesium   2.10   2.02  -0.1
    Creatinine   0.85   1.32  +0.5


### Delta direction check

In [ ]:
#RDW
total_deltas = pd.read_csv('total_deltas.csv')
rdw_data = total_deltas[total_deltas['feature'] == 'lab_51277_daily']

rdw_normal = rdw_data[
    (rdw_data['mean_current_value'] >= 10.5) & 
    (rdw_data['mean_current_value'] <= 15.5)
]
rdw_normal['position'] = 'below_optimum'
rdw_normal.loc[rdw_normal['mean_current_value'] > 13.0, 'position'] = 'above_optimum'
rdw_normal.loc[rdw_normal['mean_current_value'] == 13.0, 'position'] = 'at_optimum'

print("Delta by position relative to optimum (13.0):")
print(rdw_normal.groupby('position').agg({
    'delta': ['mean', 'count', 'std'],
    'mean_current_value': 'mean'
}).round(4))

Delta by position relative to optimum (13.0):
                delta                mean_current_value
                 mean  count     std               mean
position                                               
above_optimum  0.0212  13601  0.0237            14.1031
at_optimum     0.0004    307  0.0020            13.0000
below_optimum -0.0081   3925  0.0108            12.5054


In [36]:
hf_data = total_deltas[total_deltas['feature'] == 'ccsr_CIR019']

hf_data['group'] = 'without_diagnosis'
hf_data.loc[hf_data['mean_current_value'] == 1, 'group'] = 'with_diagnosis'

print("Delta by diagnosis presence:")
print(hf_data.groupby('group').agg({
    'delta': ['mean', 'count', 'std'],
    'mean_current_value': 'mean'
}).round(4))

Delta by diagnosis presence:
                    delta                mean_current_value
                     mean  count     std               mean
group                                                      
with_diagnosis    -0.0147  13638  0.0167                1.0
without_diagnosis  0.0151  67978  0.0157                0.0


In [40]:
gender_data = total_deltas[total_deltas['feature'] == 'gender_male']
gender_data['group'] = 'female'
gender_data.loc[gender_data['mean_current_value'] == 1, 'group'] = 'male'

result = gender_data.groupby('group').agg({
    'delta': ['mean', 'count', 'std'],
    'mean_current_value': 'mean',
    'risk_actual': 'mean'
}).round(4)
print(result)

         delta                mean_current_value risk_actual
          mean  count     std               mean        mean
group                                                       
female -0.0079  43488  0.0121                0.0      0.4243
male    0.0078  38128  0.0101                1.0      0.4803


In [ ]:
#Hemoglobin
rdw_data = total_deltas[total_deltas['feature'] == 'lab_51222_daily']

rdw_normal = rdw_data[
    (rdw_data['mean_current_value'] >= 13.7) & 
    (rdw_data['mean_current_value'] <= 17.5)
]
rdw_normal['position'] = 'below_optimum'
rdw_normal.loc[rdw_normal['mean_current_value'] > 15.6, 'position'] = 'above_optimum'
rdw_normal.loc[rdw_normal['mean_current_value'] == 15.6, 'position'] = 'at_optimum'

print("Delta by position relative to optimum (15.6):")
print(rdw_normal.groupby('position').agg({
    'delta': ['mean', 'count', 'std'],
    'mean_current_value': 'mean'
}).round(4))

Delta by position relative to optimum (15.6):
                delta               mean_current_value
                 mean count     std               mean
position                                              
above_optimum  0.0014   186  0.0035            16.1487
at_optimum     0.0003    24  0.0010            15.6000
below_optimum  0.0000  1850  0.0031            14.3523


In [43]:
#Platet Count
rdw_data = total_deltas[total_deltas['feature'] == 'lab_51265_daily']

rdw_normal = rdw_data[
    (rdw_data['mean_current_value'] >= 150) & 
    (rdw_data['mean_current_value'] <= 400)
]
rdw_normal['position'] = 'below_optimum'
rdw_normal.loc[rdw_normal['mean_current_value'] > 275, 'position'] = 'above_optimum'
rdw_normal.loc[rdw_normal['mean_current_value'] == 275, 'position'] = 'at_optimum'

print("Delta by position relative to optimum (275):")
print(rdw_normal.groupby('position').agg({
    'delta': ['mean', 'count', 'std'],
    'mean_current_value': 'mean'
}).round(4))

Delta by position relative to optimum (275):
                delta                mean_current_value
                 mean  count     std               mean
position                                               
above_optimum  0.0062   4901  0.0079           321.8059
at_optimum     0.0006     35  0.0012           275.0000
below_optimum -0.0065  13741  0.0090           208.9617


In [44]:
#Sodium
rdw_data = total_deltas[total_deltas['feature'] == 'lab_50983_daily']

rdw_normal = rdw_data[
    (rdw_data['mean_current_value'] >= 135) & 
    (rdw_data['mean_current_value'] <= 147)
]
rdw_normal['position'] = 'below_optimum'
rdw_normal.loc[rdw_normal['mean_current_value'] > 141, 'position'] = 'above_optimum'
rdw_normal.loc[rdw_normal['mean_current_value'] == 141, 'position'] = 'at_optimum'

print("Delta by position relative to optimum (141):")
print(rdw_normal.groupby('position').agg({
    'delta': ['mean', 'count', 'std'],
    'mean_current_value': 'mean'
}).round(4))

Delta by position relative to optimum (141):
                delta                mean_current_value
                 mean  count     std               mean
position                                               
above_optimum  0.0002   4328  0.0017           142.7622
at_optimum     0.0006    992  0.0016           141.0000
below_optimum  0.0077  15348  0.0087           138.1438


In [46]:
age_data = total_deltas[total_deltas['feature'] == 'age']

def get_age_group(age):
    if age < 40:
        return 'young (<40)'
    elif age < 65:
        return 'middle (40-64)'
    elif age < 80:
        return 'elderly (65-79)'
    else:
        return 'very_elderly (80+)'

age_data['age_group'] = age_data['mean_current_value'].apply(get_age_group)

result = age_data.groupby('age_group').agg({
    'delta': ['mean', 'count', 'std'],
    'mean_current_value': 'mean'
}).round(4)
print(result)       

                     delta                mean_current_value
                      mean  count     std               mean
age_group                                                   
elderly (65-79)    -0.0011  22347  0.0068            71.5722
middle (40-64)     -0.0052  30747  0.0160            53.9256
very_elderly (80+) -0.0029  14245  0.0101            86.1018
young (<40)        -0.0042  14277  0.0122            30.4860


In [48]:
#BUN
rdw_data = total_deltas[total_deltas['feature'] == 'lab_51006_daily']

rdw_normal = rdw_data[
    (rdw_data['mean_current_value'] >= 6) & 
    (rdw_data['mean_current_value'] <= 20)
]
rdw_normal['position'] = 'below_optimum'
rdw_normal.loc[rdw_normal['mean_current_value'] > 13, 'position'] = 'above_optimum'
rdw_normal.loc[rdw_normal['mean_current_value'] == 13, 'position'] = 'at_optimum'

print("Delta by position relative to optimum (13):")
print(rdw_normal.groupby('position').agg({
    'delta': ['mean', 'count', 'std'],
    'mean_current_value': 'mean'
}).round(4))

Delta by position relative to optimum (13):
                delta               mean_current_value
                 mean count     std               mean
position                                              
above_optimum  0.0010  6956  0.0023            16.4210
at_optimum     0.0001   515  0.0006            13.0000
below_optimum  0.0017  6691  0.0042             9.7989
